In [ ]:
"""
Evaluation for oof_<language>_<model>_corrected.csv (from correct_oof_predictions.py)
Three complementary views:
  1. Token-level  - is the label right per token? (ignores span boundaries)
  2. Span-level   - is the full span (start, end, label) right? (seqeval strict)
  3. Span-boundary-only - did the model find the span at all, ignoring label?
  4. BIO-prefix accuracy - B / I / O accuracy regardless of the event label

Every metric below is computed on the WHOLE set treated as one corpus -- all
sentences' tokens/spans are pooled before precision/recall/F1 are computed.
There is no per-sentence scoring or averaging anywhere in this notebook.

NOTE on file structure: this file is WORD-LEVEL -- one row per word, not
one row per sentence. There are no stringified list columns here (that was
the older predictions/test_<language>.csv format from earlier in this
project) -- 'true_label' / 'pred_label_raw' / 'pred_label_corrected' are
each a single plain label string per row, e.g. 'B-process'. Sentences are
reconstructed by grouping on (source_file, sentence_id), ordered by
word_index, before anything span-level (seqeval) can be computed -- seqeval
needs one label sequence per sentence, not a flat list of words.

We evaluate 'pred_label_corrected' against 'true_label' throughout. Swap in
'pred_label_raw' instead if you want the uncorrected model output.
"""

from itertools import chain

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from seqeval.metrics import classification_report as seq_report
from seqeval.scheme import IOB2
from sklearn.metrics import (
    classification_report as sk_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
)

# -- Load ----------------------------------------------------------------
language = "Italian"
model = "xlm-roberta-base"
path = "PATH"

df = pd.read_csv(f"{path}/oof_{language}_{model}_corrected.csv")
# columns: source_file, sentence_id, word_index, word,
#          true_label, pred_label_raw, pred_label_corrected,
#          correction_applied, correct

PRED_COL = "pred_label_corrected"  # swap for 'pred_label_raw' to check uncorrected predictions
EVENT_LABELS = ["process", "stative_event", "change_of_state", "non_event"]

print(f"Loaded {len(df)} word-level rows across {df.groupby(['source_file','sentence_id']).ngroups} sentences.")
print(f"Correction applied: {df['correction_applied'].unique().tolist()}")

In [ ]:
# -- Regroup word-level rows back into per-sentence label sequences ------
# Required for seqeval (span-level views) -- token-level views could technically
# work on the flat word list directly, but we build this once and reuse it
# everywhere below for consistency.

true_seqs, pred_seqs = [], []
for _, g in df.sort_values("word_index").groupby(["source_file", "sentence_id"], sort=False):
    true_seqs.append(g["true_label"].tolist())
    pred_seqs.append(g[PRED_COL].tolist())

# Sanity check: every sentence's true/pred sequence should be the same length
# (they always will be here, since both come from the same word-level rows --
# this just confirms the regrouping itself didn't silently drop anything)
bad = [i for i, (t, p) in enumerate(zip(true_seqs, pred_seqs)) if len(t) != len(p)]
if bad:
    print(f"WARNING: {len(bad)} sentences have mismatched true/pred lengths after regrouping -- check these.")
else:
    print(f"Regrouped into {len(true_seqs)} sentences, all lengths match.")


In [ ]:
# -- 1. TOKEN-LEVEL (sklearn) ---------------------------------------------
# Strips the B-/I- prefix and scores every token independently, ignoring
# whether spans line up.

def strip_bio(label):
    return 'O' if label == 'O' else label.split('-', 1)[1]

flat_true = [strip_bio(l) for l in chain.from_iterable(true_seqs)]
flat_pred = [strip_bio(l) for l in chain.from_iterable(pred_seqs)]

print("=" * 60)
print("1. TOKEN-LEVEL CLASSIFICATION (per token, ignores span boundaries)")
print("=" * 60)
print(sk_report(
    flat_true, flat_pred,
    labels=EVENT_LABELS + ["O"],
    zero_division=0
))


In [ ]:
# -- 2. SPAN-LEVEL (seqeval) ------------------------------------------------
# Counts a span correct only if BOTH the boundaries AND the label match exactly.

report_dict = seq_report(
    true_seqs, pred_seqs,
    scheme=IOB2,
    digits=3,
    zero_division=0,
    output_dict=True   # <-- key change: returns a dict instead of a printed string
)

df = pd.DataFrame(report_dict)

new_order = ["non_event", "stative_event", "process", "change_of_state", "weighted avg"]
df = df[new_order]

# optional: round for cleaner LaTeX output
df = df.round(3)

print(df.to_latex(float_format="%.3f"))

In [ ]:
# -- 3. SPAN-BOUNDARY-ONLY --------------------------------------------------
# Did the model find the span at all, regardless of which label it assigned?
# Collapses all event types to a single "EVENT" class.

def flat_to_bio_binary(labels):
    """Collapse all event types to EVENT, keep BIO prefix, O stays O."""
    return [
        "O" if l == "O" else l.split("-")[0] + "-EVENT"
        for l in labels
    ]

true_bin = [flat_to_bio_binary(seq) for seq in true_seqs]
pred_bin = [flat_to_bio_binary(seq) for seq in pred_seqs]

report_dict = seq_report(
    true_bin, pred_bin,
    scheme=IOB2,
    digits=4,
    zero_division=0,
    output_dict=True
)

df_binary = pd.DataFrame(report_dict).transpose()
df_binary['support'] = df_binary['support'].astype(int)

print(df_binary.to_latex(float_format="%.3f"))


In [ ]:
# -- 4. BIO-PREFIX ACCURACY --------------------------------------------------
# Plain B / I / O accuracy per token, ignoring the event-type suffix entirely.

def strip_suffix(label):
    return "O" if label == "O" else label.split("-")[0]

true_bio_flat = [strip_suffix(l) for l in chain.from_iterable(true_seqs)]
pred_bio_flat = [strip_suffix(l) for l in chain.from_iterable(pred_seqs)]

print("=" * 60)
print("4. SPAN-BOUNDARY ONLY (B / I / O prefix accuracy)")
print("=" * 60)
print(sk_report(true_bio_flat, pred_bio_flat, labels=["B", "I", "O"], zero_division=0))


In [ ]:
# -- Optional: confusion matrix at token level (event labels only) ---------
cm_labels = EVENT_LABELS + ["O"]
cm = confusion_matrix(flat_true, flat_pred, labels=cm_labels)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=cm_labels)
fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title(f"Token-level confusion matrix - {language} ({model})")
plt.tight_layout()
plt.savefig(f"confusion_matrix_{language}_{model}.png", dpi=150)
plt.show()
